# Model 4 — CatBoost for FD001 Remaining Useful Life

This experiment applies CatBoost regression to NASA C-MAPSS FD001. It compares a raw-sensor baseline with a regularized model using causal degradation features, then evaluates the frozen method on the official test set. The split, cutoffs, targets, and metrics match Model 3 so the results are directly comparable.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import catboost
from catboost import CatBoostError, CatBoostRegressor

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_rul.catboost_modeling import (
    catboost_baseline_parameters, catboost_improved_parameters, resolve_catboost_device,
)
from cmapss_rul.data import load_fd001, make_validation_subset, split_by_engine
from cmapss_rul.evaluation import regression_metrics
from cmapss_rul.features import engineer_features, nonconstant_sensor_columns, select_sensors

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'cmapss' / 'CMaps'
FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
for directory in (FIGURE_DIR, MODEL_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE, RUL_CAP = 42, 125
print(f'CatBoost {catboost.__version__} | project: {PROJECT_ROOT}')

## 1. Load FD001 and visualize the data

Training engines run until failure, while test engines stop earlier. The first figures show lifetime variation and how informative sensors move as one engine approaches failure.

In [ ]:
train_df, test_df, official_test = load_fd001(DATA_DIR)
summary = pd.DataFrame({
    'rows': [len(train_df), len(test_df)],
    'engines': [train_df.unit_id.nunique(), test_df.unit_id.nunique()],
    'longest_history': [train_df.cycle.max(), test_df.cycle.max()],
}, index=['train', 'test'])
display(summary)
display(train_df.head())

In [ ]:
lifetimes = train_df.groupby('unit_id').cycle.max()
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(lifetimes, bins=18, kde=True, color='#6A4C93', ax=ax)
ax.axvline(lifetimes.mean(), color='#E76F51', linestyle='--', label=f'Mean = {lifetimes.mean():.1f}')
ax.set(title='FD001 engine lifetime distribution', xlabel='Lifetime (cycles)', ylabel='Engines')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'catboost_fd001_lifetimes.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
engine = train_df.loc[train_df.unit_id.eq(1)].copy()
shown_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_12', 'sensor_15', 'sensor_21']
scaled = engine[shown_sensors].apply(lambda column: (column - column.mean()) / column.std())
scaled['cycle'] = engine.cycle.to_numpy()
long_sensors = scaled.melt('cycle', var_name='sensor', value_name='standardized_value')
fig, ax = plt.subplots(figsize=(11, 6))
sns.lineplot(data=long_sensors, x='cycle', y='standardized_value', hue='sensor', linewidth=1.3, ax=ax)
ax.set(title='Engine 1 sensor degradation trajectories', ylabel='Standardized sensor value')
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'catboost_fd001_sensor_trends.png', dpi=160, bbox_inches='tight')
plt.show()

## 2. Reproduce the Model 3 validation protocol

The same random seed keeps exactly 80 engines for fitting and 20 different engines for validation. Validation trajectories are cut before failure, preventing an unrealistic all-zero last-cycle target.

In [ ]:
train_split, validation_full = split_by_engine(train_df, validation_size=0.20, random_state=RANDOM_STATE)
validation_history, validation_snapshots = make_validation_subset(
    validation_full, min_rul=10, max_rul=100, random_state=RANDOM_STATE
)
assert set(train_split.unit_id).isdisjoint(set(validation_history.unit_id))
device = resolve_catboost_device(prefer_gpu=True)
print(f'CatBoost device selected: {device}')
print(f'Train engines: {train_split.unit_id.nunique()} | validation engines: {validation_history.unit_id.nunique()}')
display(validation_snapshots[['unit_id', 'cycle', 'rul']].head())

## 3. Baseline CatBoost

The baseline sees cycle, settings, and all non-constant raw sensors. If full GPU training fails after the capability check, this cell transparently retries on CPU.

In [ ]:
raw_sensors = nonconstant_sensor_columns(train_split)
baseline_columns = ['cycle', 'setting_1', 'setting_2', 'setting_3', *raw_sensors]
baseline_model = CatBoostRegressor(**catboost_baseline_parameters(device))
baseline_start = time.perf_counter()
try:
    baseline_model.fit(train_split[baseline_columns], train_split.rul)
except CatBoostError as error:
    if device != 'GPU':
        raise
    print(f'GPU training failed; retrying on CPU: {error}')
    device = 'CPU'
    baseline_model = CatBoostRegressor(**catboost_baseline_parameters(device))
    baseline_model.fit(train_split[baseline_columns], train_split.rul)
baseline_train_seconds = time.perf_counter() - baseline_start
baseline_predict_start = time.perf_counter()
baseline_predictions = np.clip(baseline_model.predict(validation_snapshots[baseline_columns]), 0, None)
baseline_predict_seconds = time.perf_counter() - baseline_predict_start
baseline_metrics = regression_metrics(validation_snapshots.rul, baseline_predictions)
display(pd.DataFrame([baseline_metrics], index=['CatBoost baseline']).round(3))

## 4. Improved CatBoost

Training-only sensor selection, a 125-cycle target cap, rolling means and variation, recent changes, degradation slopes, stronger regularization, and overfitting detection form the improved pipeline.

In [ ]:
selection_frame = train_split.assign(rul=train_split.rul.clip(upper=RUL_CAP))
selected_sensors = select_sensors(selection_frame, top_n=8)
X_train = engineer_features(train_split, selected_sensors, windows=(5, 15))
X_valid_history = engineer_features(validation_history, selected_sensors, windows=(5, 15))
last_indices = validation_history.groupby('unit_id').cycle.idxmax()
X_valid_last = X_valid_history.loc[last_indices].reset_index(drop=True)
y_train = train_split.rul.clip(upper=RUL_CAP)
y_valid_history = validation_history.rul.clip(upper=RUL_CAP)
print('Selected sensors:', selected_sensors)

improved_model = CatBoostRegressor(**catboost_improved_parameters(device))
improved_start = time.perf_counter()
improved_model.fit(X_train, y_train, eval_set=(X_valid_history, y_valid_history))
improved_train_seconds = time.perf_counter() - improved_start
improved_predict_start = time.perf_counter()
improved_predictions = np.clip(improved_model.predict(X_valid_last), 0, None)
improved_predict_seconds = time.perf_counter() - improved_predict_start
improved_metrics = regression_metrics(validation_snapshots.rul, improved_predictions)
print(f'Best iteration: {improved_model.get_best_iteration()}')
display(pd.DataFrame([improved_metrics], index=['CatBoost improved']).round(3))

In [ ]:
metric_rows = [
    {'model': 'CatBoost baseline', 'split': 'validation', 'device': device, 'train_seconds': baseline_train_seconds, 'predict_seconds': baseline_predict_seconds, **baseline_metrics},
    {'model': 'CatBoost improved', 'split': 'validation', 'device': device, 'train_seconds': improved_train_seconds, 'predict_seconds': improved_predict_seconds, **improved_metrics},
]
comparison = pd.DataFrame(metric_rows)
plot_data = comparison.melt(id_vars='model', value_vars=['rmse', 'mae'], var_name='metric', value_name='cycles')
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=plot_data, x='metric', y='cycles', hue='model', palette=['#888888', '#6A4C93'], ax=ax)
ax.set(title='Validation error: baseline vs improved CatBoost', xlabel='', ylabel='Cycles (lower is better)')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'catboost_fd001_metric_comparison.png', dpi=160, bbox_inches='tight')
plt.show()
display(comparison.round(3))

In [ ]:
history = improved_model.get_evals_result()
learn_rmse = history['learn']['RMSE']
valid_rmse = history['validation']['RMSE']
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(learn_rmse, label='Training RMSE', color='#457B9D')
ax.plot(valid_rmse, label='Validation RMSE', color='#E76F51')
ax.axvline(improved_model.get_best_iteration(), linestyle='--', color='black', label='Best iteration')
ax.set(title='CatBoost learning curve', xlabel='Boosting iteration', ylabel='RMSE')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'catboost_fd001_learning_curve.png', dpi=160, bbox_inches='tight')
plt.show()

## 5. Official FD001 test evaluation

The selected method is refitted on every training engine. Test features use only each engine's observed cycles, and predictions are made at its last observed cycle.

In [ ]:
full_selection = train_df.assign(rul=train_df.rul.clip(upper=RUL_CAP))
final_sensors = select_sensors(full_selection, top_n=8)
X_full_train = engineer_features(train_df, final_sensors, windows=(5, 15))
X_full_test = engineer_features(test_df, final_sensors, windows=(5, 15))
test_last_indices = test_df.groupby('unit_id').cycle.idxmax()
X_test_last = X_full_test.loc[test_last_indices].reset_index(drop=True)
final_params = catboost_improved_parameters(device)
for key in ('use_best_model', 'od_type', 'od_wait'):
    final_params.pop(key)
final_params['iterations'] = max(1, improved_model.get_best_iteration() + 1)
final_model = CatBoostRegressor(**final_params)
final_start = time.perf_counter()
final_model.fit(X_full_train, train_df.rul.clip(upper=RUL_CAP))
final_train_seconds = time.perf_counter() - final_start
test_predict_start = time.perf_counter()
test_predictions = np.clip(final_model.predict(X_test_last), 0, None)
test_predict_seconds = time.perf_counter() - test_predict_start
official_metrics = regression_metrics(official_test.rul, test_predictions)
metric_rows.append({'model': 'CatBoost improved', 'split': 'official_test', 'device': device, 'train_seconds': final_train_seconds, 'predict_seconds': test_predict_seconds, **official_metrics})
display(pd.DataFrame([official_metrics], index=['Official FD001 test']).round(3))

In [ ]:
test_results = pd.DataFrame({'unit_id': official_test.unit_id.astype(int), 'true_rul': official_test.rul, 'predicted_rul': test_predictions})
test_results['residual'] = test_results.predicted_rul - test_results.true_rul
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
limit = max(test_results.true_rul.max(), test_results.predicted_rul.max()) + 5
sns.scatterplot(data=test_results, x='true_rul', y='predicted_rul', hue='residual', palette='coolwarm', s=58, ax=axes[0])
axes[0].plot([0, limit], [0, limit], '--', color='black')
axes[0].set(title='Official test: true vs predicted RUL', xlim=(0, limit), ylim=(0, limit))
sns.histplot(test_results.residual, bins=16, kde=True, color='#6A4C93', ax=axes[1])
axes[1].axvline(0, color='#E76F51', linestyle='--')
axes[1].set(title='Official-test residuals', xlabel='Prediction − true RUL')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'catboost_fd001_predictions.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
importance = pd.Series(final_model.get_feature_importance(), index=X_full_train.columns).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=importance.values, y=importance.index, color='#6A4C93', ax=ax)
ax.set(title='Top 15 CatBoost feature importances', xlabel='Importance', ylabel='Feature')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'catboost_fd001_feature_importance.png', dpi=160, bbox_inches='tight')
plt.show()
display(importance.to_frame('importance').round(3))

## 6. Compare Models 3 and 4, then save results

In [ ]:
catboost_metrics = pd.DataFrame(metric_rows)
xgboost_metrics = pd.read_csv(REPORT_DIR / 'xgboost_fd001_metrics.csv')
official_comparison = pd.concat([
    xgboost_metrics.query("split == 'official_test'"),
    catboost_metrics.query("split == 'official_test'"),
])
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=official_comparison, x='model', y='rmse', hue='model', palette=['#2A9D8F', '#6A4C93'], legend=False, ax=ax)
ax.set(title='Official FD001 RMSE: Models 3 and 4', xlabel='', ylabel='RMSE cycles (lower is better)')
ax.tick_params(axis='x', rotation=10)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'catboost_fd001_vs_xgboost.png', dpi=160, bbox_inches='tight')
plt.show()

catboost_metrics.to_csv(REPORT_DIR / 'catboost_fd001_metrics.csv', index=False)
test_results.to_csv(REPORT_DIR / 'catboost_fd001_predictions.csv', index=False)
final_model.save_model(MODEL_DIR / 'catboost_fd001.cbm')
improvement = baseline_metrics['rmse'] - improved_metrics['rmse']
print(f'Engineered CatBoost validation RMSE change: {improvement:+.2f} cycles (positive is better).')
print(f'Official CatBoost RMSE: {official_metrics["rmse"]:.2f} cycles')
display(official_comparison[['model', 'device', 'rmse', 'mae', 'r2', 'nasa_score']].round(3))